# BirdCLEF+ 2026 — Ensemble Submission: Perch+MLP + SED

**兩路 ensemble**: Perch v2 embedding → MLP head (Route B baseline) + EfficientNet-B0 SED (mel spectrogram)

## Kaggle Notebook 設定
- **Accelerator:** None (CPU only)
- **Internet:** OFF
- **Inputs** (五個):
  1. Competition: **BirdCLEF+ 2026** → `/kaggle/input/birdclef-2026/`
  2. Dataset: `perch-hoplite-wheels` → `/kaggle/input/perch-hoplite-wheels/`
  3. Model: `perch-v2-cpu` → `/kaggle/input/perch-v2-cpu/`
  4. Dataset: `birdclef-mlp-head` → `/kaggle/input/birdclef-mlp-head/` (含 `mlp_best.pt` + `sed_best.pt`)
  5. (可選) Dataset: `timm-wheels` → 如果 Kaggle 沒預裝 timm

In [ ]:
WHEELS = "/kaggle/input/perch-hoplite-wheels"
!pip install -q --no-index --find-links {WHEELS} "tensorflow==2.20.0" perch-hoplite
# timm 通常 Kaggle 已預裝；若沒有，需另外準備 wheels
try:
    import timm
    print(f"timm={timm.__version__} (pre-installed)")
except ImportError:
    !pip install -q timm
    import timm
    print(f"timm={timm.__version__} (just installed)")

In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["CUDA_VISIBLE_DEVICES"] = ""
os.environ["KAGGLEHUB_CACHE"] = "/kaggle/input"

import time, gc
from pathlib import Path
import numpy as np, pandas as pd
import soundfile as sf
import tensorflow as tf
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
import torchaudio
from tqdm.auto import tqdm

print("TF   :", tf.__version__)
print("Torch:", torch.__version__)
print("timm :", timm.__version__)
assert tf.__version__.startswith("2.20"), f"TF {tf.__version__}, 要 2.20"

# ── 路徑 ──
DATA_ROOT = Path("/kaggle/input/birdclef-2026")
PERCH_DIR = Path("/kaggle/input/perch-v2-cpu")
HEAD_DIR  = Path("/kaggle/input/birdclef-mlp-head")
MLP_CKPT  = HEAD_DIR / "mlp_best.pt"
SED_CKPT  = HEAD_DIR / "sed_best.pt"
OUT_CSV   = Path("/kaggle/working/submission.csv")

# ── 常數 ──
SR, FRAME_LEN, BATCH = 32000, 160000, 32
N_CLASSES, EMB_DIM, SEGS_PER_FILE = 234, 1536, 12

# ── Ensemble 權重 ──
MLP_WEIGHT = 0.7
SED_WEIGHT = 0.3

# ── Taxonomy ──
taxonomy_df = pd.read_csv(DATA_ROOT / "taxonomy.csv")
species_cols = taxonomy_df["primary_label"].tolist()
sample = pd.read_csv(DATA_ROOT / "sample_submission.csv", nrows=1)
sample_species = [c for c in sample.columns if c != "row_id"]
assert len(species_cols) == N_CLASSES
assert set(sample_species) == set(species_cols)

test_files = sorted((DATA_ROOT / "test_soundscapes").glob("*.ogg"))
print(f"test files: {len(test_files)}, species: {len(species_cols)}")

## Model 1: Perch v2 + MLP Head

In [ ]:
class MLPHead(nn.Module):
    def __init__(self, in_dim=EMB_DIM, n_classes=N_CLASSES, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 768), nn.BatchNorm1d(768), nn.ReLU(inplace=True), nn.Dropout(dropout),
            nn.Linear(768, 384), nn.ReLU(inplace=True), nn.Dropout(dropout),
            nn.Linear(384, n_classes),
        )
    def forward(self, x): return self.net(x)

mlp_head = MLPHead()
mlp_head.load_state_dict(torch.load(MLP_CKPT, map_location="cpu"))
mlp_head.eval()
print(f"MLP head: {sum(p.numel() for p in mlp_head.parameters()):,} params")

In [ ]:
from perch_hoplite.zoo import model_configs

t0 = time.time()
perch = model_configs.load_model_by_name("perch_v2_cpu")
print(f"Perch loaded in {time.time()-t0:.1f}s, SR={perch.sample_rate}")
assert perch.sample_rate == SR and perch.batchable

t0 = time.time()
_ = perch.batch_embed(np.zeros((BATCH, FRAME_LEN), dtype="float32"))
print(f"Perch warm-up in {time.time()-t0:.1f}s")

## Model 2: SED (EfficientNet-B0 + AttBlockV2)

In [ ]:
# ── Mel spectrogram ──
N_MELS, N_FFT, HOP_LENGTH, F_MIN, F_MAX = 128, 2048, 512, 20, 16000

class MelSpecTransform(nn.Module):
    def __init__(self):
        super().__init__()
        self.mel = torchaudio.transforms.MelSpectrogram(
            sample_rate=SR, n_fft=N_FFT, hop_length=HOP_LENGTH,
            n_mels=N_MELS, f_min=F_MIN, f_max=F_MAX, power=2.0,
        )
        self.db = torchaudio.transforms.AmplitudeToDB(top_db=80)
    def forward(self, waveform):
        spec = self.mel(waveform)
        spec = self.db(spec)
        spec = (spec + 80) / 80
        return spec

# ── AttBlockV2 ──
def init_layer(layer):
    nn.init.xavier_uniform_(layer.weight)
    if hasattr(layer, "bias") and layer.bias is not None:
        layer.bias.data.fill_(0.0)

def init_bn(bn):
    bn.bias.data.fill_(0.0)
    bn.weight.data.fill_(1.0)

class AttBlockV2(nn.Module):
    def __init__(self, in_features, out_features, activation="sigmoid"):
        super().__init__()
        self.activation = activation
        self.att = nn.Conv1d(in_features, out_features, kernel_size=1, bias=True)
        self.cla = nn.Conv1d(in_features, out_features, kernel_size=1, bias=True)
        init_layer(self.att)
        init_layer(self.cla)
    def forward(self, x):
        norm_att = torch.softmax(torch.tanh(self.att(x)), dim=-1)
        cla = self.nonlinear_transform(self.cla(x))
        clipwise_output = torch.sum(norm_att * cla, dim=2)
        return clipwise_output, norm_att, cla
    def nonlinear_transform(self, x):
        if self.activation == "linear": return x
        elif self.activation == "sigmoid": return torch.sigmoid(x)

# ── SEDModel ──
class SEDModel(nn.Module):
    def __init__(self, backbone_name="tf_efficientnet_b0_ns", n_classes=N_CLASSES, n_mels=N_MELS):
        super().__init__()
        self.mel_transform = MelSpecTransform()
        self.bn0 = nn.BatchNorm2d(n_mels)
        init_bn(self.bn0)
        base_model = timm.create_model(backbone_name, pretrained=False, in_chans=1,
                                       drop_path_rate=0.2, drop_rate=0.5)
        layers = list(base_model.children())[:-2]
        self.encoder = nn.Sequential(*layers)
        in_features = base_model.classifier.in_features
        self.fc1 = nn.Linear(in_features, in_features, bias=True)
        init_layer(self.fc1)
        self.att_block = AttBlockV2(in_features, n_classes, activation="sigmoid")

    def forward(self, waveform):
        spec = self.mel_transform(waveform)
        x = spec.unsqueeze(1)
        x = x.transpose(1, 2)
        x = self.bn0(x)
        x = x.transpose(1, 2)
        x = self.encoder(x)
        x = torch.mean(x, dim=2)
        x1 = F.max_pool1d(x, kernel_size=3, stride=1, padding=1)
        x2 = F.avg_pool1d(x, kernel_size=3, stride=1, padding=1)
        x = x1 + x2
        x = x.transpose(1, 2)
        x = F.relu_(self.fc1(x))
        x = x.transpose(1, 2)
        norm_att = torch.softmax(torch.tanh(self.att_block.att(x)), dim=-1)
        segmentwise_logit = self.att_block.cla(x)
        clipwise_logit = torch.sum(norm_att * segmentwise_logit, dim=2)
        return torch.sigmoid(clipwise_logit)

sed_model = SEDModel(pretrained=False)
sed_state = torch.load(SED_CKPT, map_location="cpu")
sed_model.load_state_dict(sed_state, strict=False)
sed_model.eval()
print(f"SED model: {sum(p.numel() for p in sed_model.parameters()):,} params")

# Warm-up
with torch.no_grad():
    _ = sed_model(torch.randn(1, FRAME_LEN))
print("SED warm-up done")

## Inference: 雙路 Ensemble

In [ ]:
def load_audio(fp):
    audio, sr = sf.read(str(fp), dtype="float32")
    if audio.ndim > 1: audio = audio.mean(axis=1)
    assert sr == SR
    return audio

def frame_audio(audio):
    if len(audio) < FRAME_LEN:
        return [np.pad(audio, (0, FRAME_LEN - len(audio)))]
    n = len(audio) // FRAME_LEN
    return [audio[i * FRAME_LEN:(i + 1) * FRAME_LEN] for i in range(n)]

rows, all_probs = [], []
buf_audio, buf_rows = [], []

def flush():
    if not buf_audio: return
    n_valid = len(buf_audio)
    # Pad to BATCH for Perch (fixed XLA shape)
    if n_valid < BATCH:
        pad = [np.zeros(FRAME_LEN, dtype="float32")] * (BATCH - n_valid)
        batch_np = np.stack(buf_audio + pad, axis=0).astype("float32")
    else:
        batch_np = np.stack(buf_audio, axis=0).astype("float32")

    # ── Route 1: Perch → MLP ──
    out = perch.batch_embed(batch_np)
    embs = out.embeddings[:n_valid, 0, 0, :].astype("float32")
    with torch.no_grad():
        mlp_probs = torch.sigmoid(mlp_head(torch.from_numpy(embs))).numpy()

    # ── Route 2: SED ──
    batch_valid = np.stack(buf_audio, axis=0).astype("float32")
    with torch.no_grad():
        sed_probs = sed_model(torch.from_numpy(batch_valid)).numpy()

    # ── Ensemble: weighted average ──
    ensemble_probs = MLP_WEIGHT * mlp_probs + SED_WEIGHT * sed_probs

    all_probs.append(ensemble_probs)
    rows.extend(buf_rows)
    buf_audio.clear()
    buf_rows.clear()

t0 = time.time()
for fp in tqdm(test_files, desc="test_soundscapes"):
    stem = fp.stem
    try:
        audio = load_audio(fp)
        frames = frame_audio(audio)
    except Exception as e:
        print(f"[skip] {stem}: {e}")
        frames = []
    for k in range(SEGS_PER_FILE):
        end_sec = (k + 1) * 5
        buf_audio.append(frames[k] if k < len(frames) else np.zeros(FRAME_LEN, dtype="float32"))
        buf_rows.append((stem, end_sec))
        if len(buf_audio) >= BATCH:
            flush()
flush()

probs_arr = np.concatenate(all_probs, axis=0) if all_probs else np.zeros((0, N_CLASSES))
print(f"inference done in {time.time()-t0:.1f}s, probs={probs_arr.shape}")
print(f"ensemble: {MLP_WEIGHT:.0%} MLP + {SED_WEIGHT:.0%} SED")

In [ ]:
# ── 寫 submission.csv ──
assert len(rows) == probs_arr.shape[0] == len(test_files) * SEGS_PER_FILE
row_ids = [f"{stem}_{end_sec}" for stem, end_sec in rows]
df = pd.DataFrame(probs_arr, columns=species_cols)
df.insert(0, "row_id", row_ids)
df = df[["row_id"] + sample_species]
df.to_csv(OUT_CSV, index=False, float_format="%.6f")
print(f"wrote {OUT_CSV} shape={df.shape}")

# ── Sanity checks ──
check = pd.read_csv(OUT_CSV)
assert check.shape[1] == 1 + N_CLASSES
assert check.shape[0] == len(test_files) * SEGS_PER_FILE
if len(check) > 0:
    vals = check.drop(columns=["row_id"]).to_numpy()
    print(f"prob range: [{vals.min():.4f}, {vals.max():.4f}]  mean: {vals.mean():.4f}")
    assert 0 <= vals.min() and vals.max() <= 1
else:
    print("(CSV 是空的 — dry-run 時正常)")
print("\n✓ submission.csv OK")